# Model 2: Fine-tuned DistilBERT (End-to-End)

**Architecture:** DistilBERT (66M params, fine-tuned) → [CLS] embedding (768-dim) → Regression head (LayerNorm → Linear(256) → GELU → Linear(1))

**Target:** Beat both BoW DNN ($46.49) and SentenceTransformer DNN (Model 1)

**Key techniques:**
- Discriminative LR: encoder 2e-5, head 1e-4
- Mixed precision (fp16)
- Linear warmup 1000 steps
- Early stopping patience=2
- max_length=128 (covers 99.8% of summaries)

## vast.ai Setup (chỉ chạy lần đầu khi thuê máy)

Sau khi `git clone` repo và `cd` vào đúng thư mục, mở terminal trên vast.ai và chạy:

```bash
pip install uv
uv sync
```

Sau đó khởi động lại Jupyter kernel rồi chạy các cell bên dưới.

In [ ]:
from pricer.items import Item
from pricer.distilbert_model import DistilBERTRunner
from pricer.evaluator import evaluate, plot_training_history

## 1. Load Data

In [ ]:
train, val, test = Item.from_hub("SeanSunny/items_full")
print(f"Train: {len(train):,} | Val: {len(val):,} | Test: {len(test):,}")

## 2. Setup Model

Loads DistilBERT pretrained weights, creates tokenized DataLoaders, configures discriminative LR optimizer.

In [ ]:
runner = DistilBERTRunner(train, val[:1000])
runner.setup(batch_size=32)

## 3. Train

Max 5 epochs with early stopping (patience=2). Mixed precision (fp16) on CUDA. Linear warmup 1000 steps.

In [ ]:
history = runner.train(epochs=5, patience=2, warmup_steps=1000)

## 4. Training History

In [ ]:
plot_training_history(history, title="Fine-tuned DistilBERT")

## 5. Evaluate on 200 Test Samples

Using `evaluate()` from `pricer/evaluator.py` — same evaluation framework as all other models.

In [ ]:
evaluate(runner.inference, test)

## 6. Save Model Weights

In [ ]:
runner.save("distilbert_model.pth")
print("Saved to distilbert_model.pth")

# 1. Sanity check — inference trên trained runner
sample = test[0]
pred_original = runner.inference(sample)
print(f"Product: {sample.title[:60]}")
print(f"Actual:  ${sample.price:.2f}")
print(f"Predict: ${pred_original:.2f}")
print(f"Error:   ${abs(pred_original - sample.price):.2f}")
print()

# 2. Load roundtrip test — load lại từ .pth và so sánh kết quả
runner.load("distilbert_model.pth")
pred_loaded = runner.inference(sample)
diff = abs(pred_original - pred_loaded)
assert diff < 0.01, f"Load mismatch! Before=${pred_original:.2f} After=${pred_loaded:.2f}"
print(f"Load roundtrip test PASSED. Diff: ${diff:.4f}")

In [ ]:
# Quick sanity check
sample = test[0]
pred = runner.inference(sample)
print(f"Product: {sample.title[:60]}")
print(f"Actual:  ${sample.price:.2f}")
print(f"Predict: ${pred:.2f}")
print(f"Error:   ${abs(pred - sample.price):.2f}")